# Notebook 02: Auto Loader & Incremental Ingestion
**Exam Coverage**: Section 2 (ELT with Spark SQL and Python)
**Duration**: 45-60 minutes
---
## Learning Objectives
By the end of this notebook, you will be able to:
- Use Auto Loader to incrementally ingest JSON files
- Configure schema inference and evolution in Auto Loader
- Implement streaming ingestion for CSV files
- Use COPY INTO for batch incremental loads
- Handle schema inference errors and rescued data columns
- Configure checkpointing for streaming queries
---

## Section 1: Introduction to Auto Loader
Auto Loader is an optimized file ingestion mechanism that incrementally processes new data files as they arrive in cloud storage.
### Key Features
- **Incremental Processing**: Only processes new files
- **Schema Inference**: Automatically detects schema from data
- **Schema Evolution**: Adapts to schema changes over time
- **Scalability**: Efficiently handles millions of files
- **Checkpointing**: Tracks processed files for exactly-once semantics
### Auto Loader vs Traditional Approaches
| Feature | Auto Loader | spark.read | COPY INTO |
|---------|-------------|------------|----------|
| **Streaming** | Yes | No | No |
| **Incremental** | Yes | Manual | Yes |
| **Schema Evolution** | Automatic | Manual | Manual |
| **Performance** | Optimized | Scans all | Good |
| **Use Case** | Continuous | One-time | Batch incremental |
### Basic Auto Loader Pattern
```python
spark.readStream \
.format("cloudFiles") \
.option("cloudFiles.format", "json") \
.option("cloudFiles.schemaLocation", checkpoint_path) \
.load(source_path) \
.writeStream \
.option("checkpointLocation", checkpoint_path) \
.trigger(availableNow=True) \
.table(table_name)
```
Let's start by importing our configuration.

Import shared variables and configuration

In [0]:
%run ./variables

# Configuration Variables

Central configuration file for the Databricks Data Engineer Certification Lab.

**Usage**: Import this file in all notebooks to maintain consistent naming.

```python
%run ./variables
```

## Unity Catalog Configuration

## Volume Paths

## Checkpoint Locations

## Table Names

...

## Browser Types

## Locations (US Cities)

## Helper Functions

## Validation

## Display Configuration Summary

In [0]:
# Set current catalog and schema
spark.sql(f"USE CATALOG {CATALOG_NAME}")
spark.sql(f"USE SCHEMA {BRONZE_SCHEMA}")

print(f"Current Catalog: {spark.catalog.currentCatalog()}")
print(f"Current Schema: {spark.catalog.currentDatabase()}")

Current Catalog: cert_prep_catalog
Current Schema: `01_bronze`


## Section 2: Auto Loader for JSON Files (Customers)
We'll start with JSON - the most common format for semi-structured data.
### Key Auto Loader Options
| Option | Purpose |
|--------|--------|
| `cloudFiles.format` | Source file format (json, csv, parquet) |
| `cloudFiles.schemaLocation` | Where to store inferred schema |
| `cloudFiles.inferColumnTypes` | Enable type inference (vs all strings) |
| `cloudFiles.schemaEvolutionMode` | How to handle schema changes |
| `checkpointLocation` | Track processing state |
### Understanding Checkpoints
Checkpoints store:
1. **Processed file list** - Which files have been ingested
2. **Schema information** - Current inferred schema
3. **Offsets** - Position in the stream
This enables exactly-once processing.

In [0]:
# Verify source data exists
print(f"Customer landing path: {CUSTOMERS_LANDING_PATH}")
print(f"Checkpoint path: {CUSTOMERS_CHECKPOINT_PATH}")

# List files in landing zone
display(dbutils.fs.ls(CUSTOMERS_LANDING_PATH))

Customer landing path: /Volumes/cert_prep_catalog/00_landing/customers
Checkpoint path: /Volumes/cert_prep_catalog/_system/checkpoints/customers_stream


path,name,size,modificationTime
dbfs:/Volumes/cert_prep_catalog/00_landing/customers/_SUCCESS,_SUCCESS,0,1778780149000
dbfs:/Volumes/cert_prep_catalog/00_landing/customers/_committed_5643142800053259177,_committed_5643142800053259177,744,1778780148000
dbfs:/Volumes/cert_prep_catalog/00_landing/customers/_started_5643142800053259177,_started_5643142800053259177,0,1778780148000
dbfs:/Volumes/cert_prep_catalog/00_landing/customers/part-00000-tid-5643142800053259177-20ccde97-cb26-486d-ae05-a07fe9195ebe-217-1-c000.json,part-00000-tid-5643142800053259177-20ccde97-cb26-486d-ae05-a07fe9195ebe-217-1-c000.json,297829,1778780148000
dbfs:/Volumes/cert_prep_catalog/00_landing/customers/part-00001-tid-5643142800053259177-20ccde97-cb26-486d-ae05-a07fe9195ebe-213-1-c000.json,part-00001-tid-5643142800053259177-20ccde97-cb26-486d-ae05-a07fe9195ebe-213-1-c000.json,300059,1778780148000
dbfs:/Volumes/cert_prep_catalog/00_landing/customers/part-00002-tid-5643142800053259177-20ccde97-cb26-486d-ae05-a07fe9195ebe-218-1-c000.json,part-00002-tid-5643142800053259177-20ccde97-cb26-486d-ae05-a07fe9195ebe-218-1-c000.json,300493,1778780148000
dbfs:/Volumes/cert_prep_catalog/00_landing/customers/part-00003-tid-5643142800053259177-20ccde97-cb26-486d-ae05-a07fe9195ebe-214-1-c000.json,part-00003-tid-5643142800053259177-20ccde97-cb26-486d-ae05-a07fe9195ebe-214-1-c000.json,301480,1778780148000
dbfs:/Volumes/cert_prep_catalog/00_landing/customers/part-00004-tid-5643142800053259177-20ccde97-cb26-486d-ae05-a07fe9195ebe-219-1-c000.json,part-00004-tid-5643142800053259177-20ccde97-cb26-486d-ae05-a07fe9195ebe-219-1-c000.json,300672,1778780148000
dbfs:/Volumes/cert_prep_catalog/00_landing/customers/part-00005-tid-5643142800053259177-20ccde97-cb26-486d-ae05-a07fe9195ebe-215-1-c000.json,part-00005-tid-5643142800053259177-20ccde97-cb26-486d-ae05-a07fe9195ebe-215-1-c000.json,300566,1778780148000
dbfs:/Volumes/cert_prep_catalog/00_landing/customers/part-00006-tid-5643142800053259177-20ccde97-cb26-486d-ae05-a07fe9195ebe-220-1-c000.json,part-00006-tid-5643142800053259177-20ccde97-cb26-486d-ae05-a07fe9195ebe-220-1-c000.json,300138,1778780148000


---
### 🎯 EXERCISE 1: Create Auto Loader Stream for JSON
**Your task**: Build an Auto Loader stream to read JSON customer data.
**Requirements:**
- Use `.readStream` with format "cloudFiles"
- Set `cloudFiles.format` to "json"
- Store schema in `CUSTOMERS_CHECKPOINT_PATH`
- Enable column type inference
- Load from `CUSTOMERS_LANDING_PATH`
- Store result in variable `customers_stream`

**Key options:**
```python
.option("cloudFiles.format", "json")
.option("cloudFiles.schemaLocation", path)
.option("cloudFiles.inferColumnTypes", "true")
```
**Hint**: Use the pattern from Section 1 introduction.

In [0]:
# TODO: Create Auto Loader stream for JSON customers

customers_stream = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format","json") \
    .option("cloudFiles.schemaLocation", CUSTOMERS_CHECKPOINT_PATH) \
    .option("cloudFiles.inferColumnTypes", "true") \
    .load(CUSTOMERS_LANDING_PATH) 

# SQL version of the above:
# CREATE OR REFRESH STREAMING TABLE customers_stream
# AS
# SELECT *
# FROM STREAM(
 # read_files(
 #   "${CUSTOMERS_LANDING_PATH}",
 #   format => "json",
 #   schemaLocation => "${CUSTOMERS_CHECKPOINT_PATH}",
 #   inferColumnTypes => "true"
 # )
# );

# SQL version of the above (directly to bronze and the bronze is automatically a delta format):
# CREATE OR REFRESH STREAMING TABLE customers_bronze_table
# AS
# SELECT *
# FROM STREAM(
# read_files(
#    "${CUSTOMERS_LANDING_PATH}",
#    format => "json",
#    schemaLocation => "${CUSTOMERS_CHECKPOINT_PATH}",
#    inferColumnTypes => "true"
#  )
#);

# Print schema to verify
customers_stream.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- location: string (nullable = true)
 |-- loyalty_tier: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- registration_date: string (nullable = true)
 |-- _rescued_data: string (nullable = true)



---
**Solution below** ⬇️

In [0]:
# ✅ SOLUTION: Auto Loader for JSON Customers

# customers_stream = spark.readStream \
#     .format("cloudFiles") \
#     .option("cloudFiles.format", "json") \
#     .option("cloudFiles.schemaLocation", CUSTOMERS_CHECKPOINT_PATH) \
#     .option("cloudFiles.inferColumnTypes", "true") \
#     .load(CUSTOMERS_LANDING_PATH)

# customers_stream.printSchema()

---
### 🎯 EXERCISE 2: Write Stream to Bronze Table
**Your task**: Write the streaming DataFrame to a Delta table.
**Requirements:**
- Format: "delta"
- Output mode: "append"
- Checkpoint: `CUSTOMERS_CHECKPOINT_PATH`
- Trigger: `availableNow=True` (process all available data then stop)
- Target table: `CUSTOMERS_BRONZE_TABLE`
- Store query in `customers_query`
- Wait for completion with `.awaitTermination()`

**Syntax:**
```python
stream.writeStream \
.format("delta") \
.outputMode("append") \
.option("checkpointLocation", ...)
.trigger(availableNow=True) \
.table(...)
```
**Hint**: Use `trigger(availableNow=True)` for micro-batch processing.

In [0]:
# TODO: Write streaming data to bronze table

customers_query = customers_stream.writeStream \
   . format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", CUSTOMERS_CHECKPOINT_PATH) \
    .trigger(availableNow=True) \
    .table(CUSTOMERS_BRONZE_TABLE)

# SQL version of the above:
# CREATE OR REFRESH STREAMING TABLE customers_bronze_table
# AS
# SELECT *
# FROM STREAM(customers_stream);

# Wait for stream to complete
customers_query.awaitTermination()

print(f"Data loaded into: {CUSTOMERS_BRONZE_TABLE}")

Data loaded into: cert_prep_catalog.01_bronze.customers_raw


---
**Solution below** ⬇️

In [0]:
# ✅ SOLUTION: Write Stream to Bronze Table

# customers_query = customers_stream.writeStream \
#     .format("delta") \
#     .outputMode("append") \
#     .option("checkpointLocation", CUSTOMERS_CHECKPOINT_PATH) \
#     .trigger(availableNow=True) \
#     .table(CUSTOMERS_BRONZE_TABLE)

# customers_query.awaitTermination()

# print(f"✅ Data loaded into: {CUSTOMERS_BRONZE_TABLE}")

In [0]:
# Verify data was loaded
customer_count = spark.table(CUSTOMERS_BRONZE_TABLE).count()
print(f"Total customers loaded: {customer_count:,}")

# Display sample records
display(spark.table(CUSTOMERS_BRONZE_TABLE).limit(10))

Total customers loaded: 10,200


customer_id,email,first_name,last_name,location,loyalty_tier,phone,registration_date,_rescued_data
f0bd4810-2e5c-472b-8cb0-eb598c23d835,lisa.miller.3825@email.com,Lisa,Miller,"Detroit, MI, USA",Platinum,+1-530-370-7011,2025-10-09,null
df23a90d-142b-413f-b287-de7f3bea989d,patricia.martinez.3826@email.com,Patricia,Martinez,"Columbus, OH, USA",Silver,null,2026-05-03,null
ed31a79d-45f8-4322-8548-55fd68e1a989,emily.martinez.3827@email.com,Emily,Martinez,"Atlanta, GA, USA",Bronze,+1-417-676-2896,2026-03-22,null
1d3ce36e-180e-463a-acae-0b82ed0fec58,emily.rodriguez.3828@email.com,Emily,Rodriguez,"Detroit, MI, USA",Silver,+1-940-428-1215,2024-12-07,null
46f1f835-3749-4da9-a354-3e29d843562a,sarah.garcia.3829@email.com,null,Garcia,"Chicago, IL, USA",Gold,+1-336-571-9228,2024-06-11,null
5464b7a7-1238-47d1-83c3-89e1238dca38,lisa.lopez.3830@email.com,Lisa,Lopez,"San Jose, CA, USA",Bronze,+1-717-948-3017,2025-11-13,null
b7ae8894-f30b-4386-8ddb-0b298de83ebb,jennifer.garcia.3831@email.com,Jennifer,gARCIA,"San Antonio, TX, USA",Bronze,+1-271-403-5253,2025-05-27,null
c7435fb0-3fec-427c-94be-7bd95730d2df,david.garcia.3832@email.com,David,Garcia,"Miami, FL, USA",null,+1-862-614-5864,2024-08-20,null
f29239b9-749a-4038-b517-976cef2acce3,sarah.martinez.3833@email.com,sARAH,Martinez,"Denver, CO, USA",Bronze,+1-869-289-3482,2026-04-19,null
4dde769b-8bbf-4e03-8716-634b6c7c0032,jane.gonzalez.3834@email.com,Jane,Gonzalez,"Fort Worth, TX, USA",Bronze,null,2024-06-16,null


### Understanding Checkpoints
Let's examine what's in the checkpoint directory:

In [0]:
# Explore checkpoint directory structure
display(dbutils.fs.ls(CUSTOMERS_CHECKPOINT_PATH))

path,name,size,modificationTime
dbfs:/Volumes/cert_prep_catalog/_system/checkpoints/customers_stream/_schemas/,_schemas/,0,1779307179667
dbfs:/Volumes/cert_prep_catalog/_system/checkpoints/customers_stream/commits/,commits/,0,1779307179667
dbfs:/Volumes/cert_prep_catalog/_system/checkpoints/customers_stream/metadata,metadata,45,1778930339000
dbfs:/Volumes/cert_prep_catalog/_system/checkpoints/customers_stream/offsets/,offsets/,0,1779307179667
dbfs:/Volumes/cert_prep_catalog/_system/checkpoints/customers_stream/sources/,sources/,0,1779307179667


**What you see:**
- `sources/` - Tracks which files have been processed
- `offsets/` - Stream position information
- `commits/` - Transaction log for the stream
- Schema information files
This ensures exactly-once processing even if the stream fails and restarts.

**Extra Practice**

If you want, you can:

- Run the `data_generator` notebook
- Read and write again to see how it processes the new data!

## Section 3: Auto Loader for CSV with Schema Evolution
CSV files require additional configuration:
- Header row handling
- Delimiter specification
- Schema evolution strategy
### Schema Evolution Modes
| Mode | Behavior |
|------|----------|
| `addNewColumns` | Automatically add new columns (default) |
| `failOnNewColumns` | Fail the stream if schema changes |
| `rescue` | Store unparseable data in `_rescued_data` |
**Best practice**: Use `addNewColumns` for flexible data ingestion.

In [0]:
# Verify product source data
print(f"Product landing path: {PRODUCTS_LANDING_PATH}")
display(dbutils.fs.ls(PRODUCTS_LANDING_PATH))

Product landing path: /Volumes/cert_prep_catalog/00_landing/products


path,name,size,modificationTime
dbfs:/Volumes/cert_prep_catalog/00_landing/products/_SUCCESS,_SUCCESS,0,1778780152000
dbfs:/Volumes/cert_prep_catalog/00_landing/products/_committed_255199070707333339,_committed_255199070707333339,464,1778780152000
dbfs:/Volumes/cert_prep_catalog/00_landing/products/_started_255199070707333339,_started_255199070707333339,0,1778780152000
dbfs:/Volumes/cert_prep_catalog/00_landing/products/part-00000-tid-255199070707333339-8aedda91-e085-4668-bb2f-db4a31a6ca14-233-1-c000.csv,part-00000-tid-255199070707333339-8aedda91-e085-4668-bb2f-db4a31a6ca14-233-1-c000.csv,10218,1778780152000
dbfs:/Volumes/cert_prep_catalog/00_landing/products/part-00001-tid-255199070707333339-8aedda91-e085-4668-bb2f-db4a31a6ca14-229-1-c000.csv,part-00001-tid-255199070707333339-8aedda91-e085-4668-bb2f-db4a31a6ca14-229-1-c000.csv,20383,1778780152000
dbfs:/Volumes/cert_prep_catalog/00_landing/products/part-00002-tid-255199070707333339-8aedda91-e085-4668-bb2f-db4a31a6ca14-230-1-c000.csv,part-00002-tid-255199070707333339-8aedda91-e085-4668-bb2f-db4a31a6ca14-230-1-c000.csv,10300,1778780152000
dbfs:/Volumes/cert_prep_catalog/00_landing/products/part-00003-tid-255199070707333339-8aedda91-e085-4668-bb2f-db4a31a6ca14-231-1-c000.csv,part-00003-tid-255199070707333339-8aedda91-e085-4668-bb2f-db4a31a6ca14-231-1-c000.csv,20349,1778780152000
dbfs:/Volumes/cert_prep_catalog/00_landing/products/part-00004-tid-255199070707333339-8aedda91-e085-4668-bb2f-db4a31a6ca14-232-1-c000.csv,part-00004-tid-255199070707333339-8aedda91-e085-4668-bb2f-db4a31a6ca14-232-1-c000.csv,20507,1778780152000


---
### 🎯 EXERCISE 3: Auto Loader for CSV Files
**Your task**: Create an Auto Loader stream for CSV product data with schema evolution.

**Requirements:**
- Format: "cloudFiles"
- File format: CSV with headers
- Schema location: `PRODUCTS_CHECKPOINT_PATH`
- Enable type inference
- Schema evolution mode: "addNewColumns"
- CSV header option: "true"
- Load from: `PRODUCTS_LANDING_PATH`

**CSV-specific options:**
```python
.option("cloudFiles.format", "csv")
.option("header", "true")
.option("cloudFiles.schemaEvolutionMode", "addNewColumns")
```
**Hint**: Similar to JSON Auto Loader, but with CSV-specific options.

In [0]:
# TODO: Create Auto Loader stream for CSV products

products_stream = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("cloudFiles.schemaLocation", PRODUCTS_CHECKPOINT_PATH) \
    .option("cloudFiles.inferColumnTypes", "true") \
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
    .option("header", "true") \
    .load(PRODUCTS_LANDING_PATH)

# SQL version of the above:
# CREATE OR REFRESH STREAMING TABLE products_stream AS 
# SELECT * FROM STREAM
#     (read_files(
#         "${PRODUCTS_LANDING_PATH}",
#        format => "csv",
#        schemaLocation => "${PRODUCTS_CHECKPOINT_PATH}",
#        inferColumnTypes => "true",
#        schemaEvolutionMode => "addNewColumns",
#        header => "true"
#))



products_stream.printSchema()

root
 |-- category: string (nullable = true)
 |-- cost: double (nullable = true)
 |-- inventory_count: integer (nullable = true)
 |-- last_updated: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- _rescued_data: string (nullable = true)



---
**Solution below** ⬇️

In [0]:
# ✅ SOLUTION: Auto Loader for CSV Products

# products_stream = spark.readStream \
#     .format("cloudFiles") \
#     .option("cloudFiles.format", "csv") \
#     .option("cloudFiles.schemaLocation", PRODUCTS_CHECKPOINT_PATH) \
#     .option("cloudFiles.inferColumnTypes", "true") \
#     .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
#     .option("header", "true") \
#     .load(PRODUCTS_LANDING_PATH)

# products_stream.printSchema()

In [0]:
# Write products to bronze table
products_query = products_stream.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", PRODUCTS_CHECKPOINT_PATH) \
    .trigger(availableNow=True) \
    .table(PRODUCTS_BRONZE_TABLE)

products_query.awaitTermination()

print(f"✅ Data loaded into: {PRODUCTS_BRONZE_TABLE}")

✅ Data loaded into: cert_prep_catalog.01_bronze.products_raw


In [0]:
# Verify products data
product_count = spark.table(PRODUCTS_BRONZE_TABLE).count()
print(f"Total products loaded: {product_count:,}")

display(spark.table(PRODUCTS_BRONZE_TABLE).limit(10))

Total products loaded: 1,010


category,cost,inventory_count,last_updated,price,product_id,product_name,subcategory,_rescued_data
Electronics,121.42,164,2026-05-14T17:35:49.000Z,273.77,P-1758,Premium Monitor,Computers,null
Clothing,null,148,2026-05-14T17:35:49.000Z,470.27,P-1759,Deluxe Scarf,null,null
Clothing,696.32,466,2026-05-14T17:35:49.000Z,1321.9,P-1760,Classic Jacket,Kids,null
Sports & Outdoors,93.06,283,2026-05-14T17:35:49.000Z,137.24,P-1761,Performance Yoga Mat,null,null
Books,612.85,117,2026-05-14T17:35:49.000Z,978.58,P-1762,History: A Story,Comics,null
Clothing,801.1,157,2026-05-14T17:35:49.000Z,1534.36,P-1763,Ultimate Scarf,Kids,null
Home & Garden,231.94,466,2026-05-14T17:35:49.000Z,354.37,P-1764,Rustic Mirror,Decor,null
sPORTS & oUTDOORS,206.48,186,2026-05-14T17:35:49.000Z,502.5,P-1765,Ultimate Water Bottle,Team Sports,null
Electronics,613.9,214,2026-05-14T17:35:49.000Z,933.89,P-1766,Premium Tablet,Computers,null
Electronics,null,36,2026-05-14T17:35:49.000Z,334.64,P-1767,Essential Tablet Pro,null,null


## Section 4: Streaming Triggers
Auto Loader supports different trigger types for controlling when data is processed.
### Trigger Types
| Trigger | Behavior | Use Case |
|---------|----------|----------|
| `availableNow=True` | Process all available data, then stop | Micro-batch, testing |
| `processingTime='10 seconds'` | Trigger every 10 seconds | Near real-time |
| `once=True` | Process once and stop (deprecated) | Use availableNow instead |
| No trigger | Continuous processing | True streaming |

**For production**: Use `processingTime` for near real-time or no trigger for continuous.

**For this lab**: We use `availableNow=True` to avoid long-running streams.

In [0]:
# Check sales landing data
print(f"Sales landing path: {SALES_LANDING_PATH}")
display(dbutils.fs.ls(SALES_LANDING_PATH))

Sales landing path: /Volumes/cert_prep_catalog/00_landing/sales


path,name,size,modificationTime
dbfs:/Volumes/cert_prep_catalog/00_landing/sales/_SUCCESS,_SUCCESS,0,1778780175000
dbfs:/Volumes/cert_prep_catalog/00_landing/sales/_committed_187016012375518090,_committed_187016012375518090,736,1778780173000
dbfs:/Volumes/cert_prep_catalog/00_landing/sales/_committed_3634413423913501299,_committed_3634413423913501299,744,1778780175000
dbfs:/Volumes/cert_prep_catalog/00_landing/sales/_committed_4356316400236721844,_committed_4356316400236721844,744,1778780156000
dbfs:/Volumes/cert_prep_catalog/00_landing/sales/_committed_458255428166583646,_committed_458255428166583646,736,1778780171000
dbfs:/Volumes/cert_prep_catalog/00_landing/sales/_committed_5217697913066241031,_committed_5217697913066241031,744,1778780164000
dbfs:/Volumes/cert_prep_catalog/00_landing/sales/_committed_5596626546366511068,_committed_5596626546366511068,744,1778780159000
dbfs:/Volumes/cert_prep_catalog/00_landing/sales/_committed_8083424683851267525,_committed_8083424683851267525,744,1778780161000
dbfs:/Volumes/cert_prep_catalog/00_landing/sales/_committed_8457016463532102938,_committed_8457016463532102938,744,1778780166000
dbfs:/Volumes/cert_prep_catalog/00_landing/sales/_committed_8795120970909606546,_committed_8795120970909606546,744,1778780169000


---
### 🎯 EXERCISE 4: Streaming with Different Triggers
**Your task**: Create an Auto Loader stream for sales data (high-volume JSON).

**Requirements:**
- Read JSON from `SALES_LANDING_PATH`
- Schema location: `SALES_CHECKPOINT_PATH`
- Enable type inference
- Write to `SALES_BRONZE_TABLE`
- Use `trigger(availableNow=True)` for this lab

**Note**: In production, you might use:
```python
.trigger(processingTime='30 seconds')  # Near real-time
# or no trigger for continuous
```
**Hint**: Same pattern as customers, but with sales paths.

In [0]:
# TODO: Create Auto Loader stream for sales JSON data

sales_stream = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "json") \
    .option("cloudFiles.schemaLocation", SALES_CHECKPOINT_PATH) \
    .option("cloudFiles.inferColumnTypes", "true") \
    .load(SALES_LANDING_PATH)

# SQL version of the above:
# CREATE OR REFRESH STREAMING TABLE sales_stream AS
# SELECT * 
# FROM STREAM(
#   read_files(
#        "${SALES_LANDING_PATH}",
#        format => "json",
#        schemaLocation => "${SALES_CHECKPOINT_PATH}",
#        inferColumnTypes => "true"
#    )
#);




sales_stream.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- line_items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- product_id: string (nullable = true)
 |    |    |-- quantity: string (nullable = true)
 |    |    |-- unit_price: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_timestamp: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- shipping_address: string (nullable = true)
 |-- _rescued_data: string (nullable = true)



---
**Solution below** ⬇️

In [0]:
# ✅ SOLUTION: Auto Loader for Sales

# sales_stream = spark.readStream \
#     .format("cloudFiles") \
#     .option("cloudFiles.format", "json") \
#     .option("cloudFiles.schemaLocation", SALES_CHECKPOINT_PATH) \
#     .option("cloudFiles.inferColumnTypes", "true") \
#     .load(SALES_LANDING_PATH)

# sales_stream.printSchema()

In [0]:
# Write sales stream to bronze table
sales_query = sales_stream.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", SALES_CHECKPOINT_PATH) \
    .trigger(availableNow=True) \
    .table(SALES_BRONZE_TABLE)

sales_query.awaitTermination()

print(f"✅ Data loaded into: {SALES_BRONZE_TABLE}")

✅ Data loaded into: cert_prep_catalog.01_bronze.sales_raw


In [0]:
# Verify sales data
sales_count = spark.table(SALES_BRONZE_TABLE).count()
print(f"Total sales loaded: {sales_count:,}")

display(spark.table(SALES_BRONZE_TABLE).limit(10))

Total sales loaded: 50,500


customer_id,line_items,order_id,order_status,order_timestamp,payment_method,shipping_address,_rescued_data
9f75077b-2c4a-4d6c-8c90-1ec698b39f0a,"List(List(P-1575, 0, 65.04), List(P-1414, 5, 276.99))",ORD-202605-01262,completed,2026-05-14T15:10:02,credit_card,"San Francisco, CA, USA",null
dc70ac6a-9d61-4dd7-b916-eb301bdbe93a,"List(List(P-1850, 2, 108.51), List(P-1235, 1, 391.26), List(P-1469, 1, 87.23))",ORD-202605-01263,pending,2026-05-14T13:22:27,null,"Philadelphia, PA, USA",null
6f75784b-a4c8-4758-8680-e3c78df5f666,"List(List(P-1485, 2, 432.7), List(P-1717, 4, 484.05))",ORD-202605-01264,completed,2026-05-14T10:29:33,credit_card,"San Antonio, TX, USA",null
51b13981-148a-445d-937d-3a2286c1c813,"List(List(P-1334, 3, 129.1), List(P-1850, 5, 302.01))",ORD-202605-01265,completed,2026-05-14T13:49:38,paypal,"Detroit, MI, USA",null
5920db9b-8334-4bd6-8cb7-0c49429b917f,"List(List(P-1850, 2, 377.74))",ORD-202605-01266,completed,2026-05-14T17:26:12,google_pay,"Los Angeles, CA, USA",null
6f75784b-a4c8-4758-8680-e3c78df5f666,"List(List(P-1363, 3, 271.5), List(P-1298, 1, 348.22), List(P-1427, 1, 151.36))",ORD-202605-01267,pending,2026-05-14T12:23:38,credit_card,"Portland, OR, USA",null
b8cb70ae-4eab-4aba-9be2-f6603f3b7689,"List(List(P-1279, 2, 388.72), List(P-1767, 1, 54.06))",ORD-202605-01268,completed,2026-05-14T07:39:16,debit_card,"San Francisco, CA, USA",null
ab84b814-f32f-45a2-94c9-fea4a3f8e247,"List(List(P-1397, 2, 273.14), List(P-1298, 3, 426.01))",ORD-202605-01269,completed,2026-05-14T09:45:01,credit_card,"Philadelphia, PA, USA",null
9cc5f0cc-7d44-4916-a818-008b0a9f94cf,"List(List(P-1293, 2, 205.19), List(P-1850, 1, 424.21))",ORD-202605-01270,pending,2026-05-14T08:06:30,debit_card,null,null
f1c3151b-37e0-429c-b10e-9553ced4d1ef,"List(List(P-1850, 4, 348.63), List(P-1267, 1, 119.58))",ORD-202605-01271,completed,2026-05-14T05:19:49,credit_card,"Philadelphia, PA, USA",null


## Section 5: COPY INTO for Batch Incremental Loads
COPY INTO is a SQL command for idempotent, incremental batch loading.
### COPY INTO vs Auto Loader
| Feature | COPY INTO | Auto Loader |
|---------|-----------|-------------|
| **Type** | Batch (SQL) | Streaming (Python/SQL) |
| **Idempotent** | Yes | Yes |
| **Checkpointing** | Automatic | Manual setup |
| **Use Case** | Scheduled batch jobs | Continuous ingestion |
| **Complexity** | Simpler | More features |
### COPY INTO Syntax
```sql
COPY INTO target_table
FROM source_path
FILEFORMAT = JSON
FORMAT_OPTIONS ('mergeSchema' = 'true')
```
**Key benefit**: Safe to re-run - automatically skips already loaded files.

---
### 🎯 EXERCISE 5: Use COPY INTO for Events
**Part 1**: Create the target table
**Requirements:**
- Table name: `{EVENTS_BRONZE_TABLE}`
- Format: DELTA
- Columns: event_id, customer_id, event_type, product_id, event_timestamp, device_type, browser, session_id, page_url
- All columns: STRING type (except event_timestamp: TIMESTAMP)

**SQL Syntax:**
```sql
CREATE TABLE IF NOT EXISTS table_name (
column1 TYPE,
column2 TYPE
)
USING DELTA
```

### NOTES BEFORE CONTINUING:

- The JSON schema in the exercise differs slightly from the provided solution schema.
- The `event_timestamp` field is provided as a string in the source JSON. While it can be ingested as a TIMESTAMP in Delta tables, COPY INTO fails due to implicit casting during schema reconciliation.
- In production Databricks pipelines, this issue is typically avoided by using Auto Loader (cloudFiles), which handles schema inference and evolution more robustly than COPY INTO.
- COPY INTO is designed for idempotent file ingestion rather than strict schema enforcement or complex type handling.
- For this exercise, I have maintained an explicit schema to reinforce structured ingestion concepts of this exercise.

In [0]:
# TODO: Create the events bronze table

spark.sql(f"""
     CREATE OR REPLACE TABLE {EVENTS_BRONZE_TABLE}
     (
         browser STRING,
         customer_id STRING,
         device_type STRING,
         event_id STRING,
         event_timestamp STRING,
         event_type STRING,
         ip_address STRING,
         page_url STRING,
         product_id STRING,
         referrer STRING,
         session_id STRING
     )
     USING DELTA
""")


print(f"Created table: {EVENTS_BRONZE_TABLE}")

Created table: cert_prep_catalog.01_bronze.events_raw


---
**Solution below** ⬇️

In [0]:
# ✅ SOLUTION: Create Events Table

# spark.sql(f"""
#     CREATE OR REPLACE TABLE {EVENTS_BRONZE_TABLE}
#     (
#         event_id STRING,
#         customer_id STRING,
#         event_type STRING,
#         product_id STRING,
#         event_timestamp STRING,
#         device_type STRING,
#         browser STRING,
#         session_id STRING,
#         page_url STRING
#     )
#     USING DELTA
# """)

# print(f"✅ Created table: {EVENTS_BRONZE_TABLE}")

---
**Part 2**: Load data with COPY INTO
**Requirements:**
- Copy from: `{EVENTS_LANDING_PATH}`
- File format: JSON
- Enable merge schema (format option)
**SQL Syntax:**
```sql
COPY INTO table_name
FROM 'path'
FILEFORMAT = JSON
FORMAT_OPTIONS ('mergeSchema' = 'true')
```

In [0]:
# TODO: Use COPY INTO to load events data

spark.sql(f"""
          COPY INTO {EVENTS_BRONZE_TABLE}
          FROM '{EVENTS_LANDING_PATH}'
          FILEFORMAT = JSON
          FORMAT_OPTIONS ('mergeSchema' = 'true')

""")

print("COPY INTO completed")

COPY INTO completed


---
**Solution below** ⬇️

In [0]:
# ✅ SOLUTION: COPY INTO for Events

spark.sql(f"""
    COPY INTO {EVENTS_BRONZE_TABLE}
    FROM '{EVENTS_LANDING_PATH}'
    FILEFORMAT = JSON
    COPY_OPTIONS ('mergeSchema' = 'true')
""")

print("✅ COPY INTO completed")

✅ COPY INTO completed


In [0]:
# Verify events data
events_count = spark.table(EVENTS_BRONZE_TABLE).count()
print(f"Total events loaded: {events_count:,}")

display(spark.table(EVENTS_BRONZE_TABLE).limit(10))

Total events loaded: 103,218


browser,customer_id,device_type,event_id,event_timestamp,event_type,ip_address,page_url,product_id,referrer,session_id
Edge,d1e834ba-dd34-4a31-8149-37e5d2da391a,tablet,e305af01-86f8-44b1-b182-95840ea10d7b,2026-05-14T11:27:52.007,view_product,192.168.32.xxx,/products/P-1255,P-1255,https://google.com/search,sess_99ad22e5d5eb
Edge,d1e834ba-dd34-4a31-8149-37e5d2da391a,tablet,02356082-98ee-4d06-9011-8d612b555ff5,2026-05-14T12:41:53.744,remove_from_cart,192.168.215.xxx,/products/P-1419,P-1419,https://google.com/search,sess_99ad22e5d5eb
Edge,d1e834ba-dd34-4a31-8149-37e5d2da391a,tablet,e8714ea5-fb00-4bee-9c13-dff3f9fa3d9f,2026-05-14T19:30:55.538,add_to_cart,192.168.26.xxx,/products/P-1223,P-1223,https://google.com/search,sess_99ad22e5d5eb
Edge,d1e834ba-dd34-4a31-8149-37e5d2da391a,tablet,01b5910b-580f-41f6-9058-f75b58c3ccc2,2026-05-14T14:39:37.944,search,192.168.148.xxx,/,null,null,sess_99ad22e5d5eb
Edge,d1e834ba-dd34-4a31-8149-37e5d2da391a,tablet,cb0ed596-b6f9-4d85-90a4-a312b2b65054,2026-05-14T17:10:39.217,search,192.168.174.xxx,/,null,https://google.com/search,sess_99ad22e5d5eb
Edge,d1e834ba-dd34-4a31-8149-37e5d2da391a,tablet,92b540ff-2727-43f6-8df5-3e5e1c9f3d0e,2026-05-14T19:14:50.466,checkout_start,192.168.103.xxx,/,null,null,sess_99ad22e5d5eb
Edge,d1e834ba-dd34-4a31-8149-37e5d2da391a,tablet,4c8886b9-48f6-46f3-b00e-b1f9f12eb034,2026-05-14T10:06:11.496,checkout_complete,192.168.33.xxx,/products/P-1235,P-1235,null,sess_99ad22e5d5eb
Edge,d1e834ba-dd34-4a31-8149-37e5d2da391a,tablet,b0672693-1c38-45f4-8ad6-f45e08b0df23,2026-05-14T16:43:42.199,remove_from_cart,192.168.245.xxx,/products/P-1210,P-1210,https://google.com/search,sess_99ad22e5d5eb
Edge,d1e834ba-dd34-4a31-8149-37e5d2da391a,tablet,f0526ce5-66ed-4fb0-8d0a-fc2dc4731a9c,2026-05-14T14:45:34.747,search,192.168.132.xxx,/,null,null,sess_99ad22e5d5eb
Opera,e8120999-c396-44dd-8b78-cea7ecc8ea50,mobile,80410adb-3876-4373-8866-0388a7c738fa,2026-05-14T14:40:48.378,view_product,192.168.7.xxx,/products/P-1850,P-1850,null,sess_18e2d0976ac4


### Test COPY INTO Idempotency
Let's verify that COPY INTO skips already-loaded files:

In [0]:
# Re-run COPY INTO - should skip already loaded files
result = spark.sql(f"""
    COPY INTO {EVENTS_BRONZE_TABLE}
    FROM '{EVENTS_LANDING_PATH}'
    FILEFORMAT = JSON
""")

display(result)

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
0,0,0


**Observation**: Notice `num_affected_rows` is 0 on the second run. COPY INTO tracks loaded files and skips them, making it safe for scheduled jobs.

## Section 6: Schema Evolution and Rescued Data
Real-world data is messy. Auto Loader provides mechanisms to handle issues:
### Schema Evolution Modes
1. **addNewColumns** (recommended)
- Automatically adds new columns when detected
- Existing queries continue to work
2. **failOnNewColumns**
- Fails the stream if schema changes
- Use when strict schema control is needed
3. **rescue**
- Stores unparseable data in `_rescued_data` column
- Prevents data loss from parsing errors
### The _rescued_data Column
When using rescue mode:
```python
.option("cloudFiles.schemaEvolutionMode", "rescue")
.option("cloudFiles.rescuedDataColumn", "_rescued_data")
```
Unparseable records are saved in `_rescued_data` instead of being dropped.

In [0]:
# Example: Rescue mode pattern (demo only, not executed)

rescue_pattern = """
spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "json") \
    .option("cloudFiles.schemaLocation", checkpoint_path) \
    .option("cloudFiles.schemaEvolutionMode", "rescue") \
    .option("cloudFiles.rescuedDataColumn", "_rescued_data") \
    .load(source_path)
"""

print("Rescue mode pattern:")
print(rescue_pattern)

Rescue mode pattern:

spark.readStream     .format("cloudFiles")     .option("cloudFiles.format", "json")     .option("cloudFiles.schemaLocation", checkpoint_path)     .option("cloudFiles.schemaEvolutionMode", "rescue")     .option("cloudFiles.rescuedDataColumn", "_rescued_data")     .load(source_path)



In [0]:
# Check for rescued data in bronze tables

for table in [CUSTOMERS_BRONZE_TABLE, PRODUCTS_BRONZE_TABLE, SALES_BRONZE_TABLE]:
    try:
        rescued_count = spark.sql(f"""
            SELECT COUNT(*) as count
            FROM {table}
            WHERE _rescued_data IS NOT NULL
        """).collect()[0]['count']
        
        if rescued_count > 0:
            print(f"⚠️  {table}: {rescued_count} records with rescued data")
        else:
            print(f"✅ {table}: No rescued data")
    except:
        print(f"ℹ️  {table}: No _rescued_data column (clean data)")

✅ cert_prep_catalog.01_bronze.customers_raw: No rescued data
✅ cert_prep_catalog.01_bronze.products_raw: No rescued data
✅ cert_prep_catalog.01_bronze.sales_raw: No rescued data


### Best Practices for Schema Management
1. ✅ **Start with schema inference** - Let Auto Loader discover the schema
2. ✅ **Use addNewColumns mode** - Flexible and safe for most cases
3. ✅ **Monitor rescued data** - Set up alerts for `_rescued_data IS NOT NULL`
4. ✅ **Version your schemas** - Store inferred schemas in source control
5. ✅ **Test with sample data** - Validate schema before production
### Common Issues and Solutions
| Issue | Solution |
|-------|----------|
| Schema mismatch | Enable `schemaEvolutionMode = "addNewColumns"` |
| Corrupt records | Use `schemaEvolutionMode = "rescue"` |
| Type inference errors | Provide explicit schema or disable type inference |
| Performance issues | Use `cloudFiles.useIncrementalListing = true` |

## Section 7: Monitoring Streaming Queries
Production streaming pipelines need monitoring. Databricks provides several tools.

In [0]:
# List all active streaming queries
active_streams = spark.streams.active

print(f"Active streaming queries: {len(active_streams)}")
for stream in active_streams:
    print(f"  - ID: {stream.id}")
    print(f"    Name: {stream.name}")
    print(f"    Status: {stream.status}")

Active streaming queries: 0


In [0]:
# View table history to see streaming writes
display(spark.sql(f"DESCRIBE HISTORY {SALES_BRONZE_TABLE}").limit(10))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-05-16T12:45:03.000Z,73065565259832,andrew_doublard@outlook.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 3194b3f9-34b3-4087-9ca1-13a0a8e4acee, epochId -> 1, statsOnLoad -> true)",null,List(2132757352063605),d9094a69-7337-4386-a7ea-8938cef3e646,0516-105551-zd7phuj0-v2n,0,WriteSerializable,true,"Map(numRemovedFiles -> 0, conflictDetectionTimeMs -> 64, numOutputRows -> 44190, numOutputBytes -> 971701, numAddedFiles -> 1)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
1,2026-05-16T12:44:59.000Z,73065565259832,andrew_doublard@outlook.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 3194b3f9-34b3-4087-9ca1-13a0a8e4acee, epochId -> 0, statsOnLoad -> true)",null,List(2132757352063605),d9094a69-7337-4386-a7ea-8938cef3e646,0516-105551-zd7phuj0-v2n,0,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 6310, numOutputBytes -> 179753, numAddedFiles -> 1)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
0,2026-05-16T12:44:53.000Z,73065565259832,andrew_doublard@outlook.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-38d680e8-c888-4ad7-8c75-6a602ec65f7f"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-944dc0ff-6474-4086-b0d6-009ab152f454""}, statsOnLoad -> false)",null,List(2132757352063605),d9094a69-7337-4386-a7ea-8938cef3e646,0516-105551-zd7phuj0-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13


In [0]:
# Check record counts across all bronze tables
display(spark.sql(f"""
    SELECT 
        '{CUSTOMERS_BRONZE_TABLE}' as table_name,
        COUNT(*) as record_count
    FROM {CUSTOMERS_BRONZE_TABLE}
    
    UNION ALL
    
    SELECT 
        '{PRODUCTS_BRONZE_TABLE}' as table_name,
        COUNT(*) as record_count
    FROM {PRODUCTS_BRONZE_TABLE}
    
    UNION ALL
    
    SELECT 
        '{SALES_BRONZE_TABLE}' as table_name,
        COUNT(*) as record_count
    FROM {SALES_BRONZE_TABLE}
    
    UNION ALL
    
    SELECT 
        '{EVENTS_BRONZE_TABLE}' as table_name,
        COUNT(*) as record_count
    FROM {EVENTS_BRONZE_TABLE}
"""))

table_name,record_count
cert_prep_catalog.01_bronze.customers_raw,10200
cert_prep_catalog.01_bronze.products_raw,1010
cert_prep_catalog.01_bronze.sales_raw,50500
cert_prep_catalog.01_bronze.events_raw,103218


## Section 8: Summary and Checkpoint
### 🎯 Key Concepts Covered
**1. Auto Loader Basics**
- `cloudFiles` format for incremental ingestion
- Schema inference and evolution
- Checkpointing for exactly-once processing
**2. File Format Support**
- JSON: Semi-structured with automatic schema
- CSV: Requires header and delimiter options
- Pattern applies to Parquet too
**3. Streaming vs Batch**
- Auto Loader: Streaming with trigger options
- COPY INTO: Batch incremental with idempotency
- Triggers: `availableNow`, `processingTime`, continuous
**4. Error Handling**
- Schema evolution: addNewColumns, failOnNewColumns, rescue
- Rescued data column for unparseable records
- Monitoring and alerting strategies
**5. Production Patterns**
- Checkpoint management
- Schema versioning
- Query monitoring
### ✅ Exam Checklist
Can you:
- [ ] Write Auto Loader code for JSON and CSV?
- [ ] Configure schema inference and evolution?
- [ ] Explain checkpointing and its importance?
- [ ] Compare Auto Loader vs COPY INTO?
- [ ] Handle schema errors with rescue mode?
- [ ] Choose appropriate trigger types?
### 📚 Next Steps
**Notebook 03** covers:
- Bronze to Silver transformations
- Data quality checks
- Deduplication strategies
- SCD Type 2 patterns
---
**🎉 Notebook Complete!**
You've mastered Auto Loader and incremental ingestion. All bronze tables are populated. Proceed to Notebook 03 to transform this data into the Silver layer.